# Lecture 16 — Flow Matching: Velocity Fields & Simulation-Free CNFs

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

This notebook builds up **flow matching** with almost no prior background — just basic Python and the
idea that a *neural network is a function with tunable knobs.* We'll lean on a single picture
the whole way through:

> **Flow matching learns a "wind" that blows scattered random dots until they arrange themselves into a target shape.**

By the end you will:

1. Understand, in plain words, **what problem** generative models solve and **why you'd care**.
2. See the **one idea** behind flow matching (a wind field) and the **one trick** for learning it (straight lines).
3. Write the **entire training loop in ~6 lines** and use it to **generate brand-new smiley faces**.
4. Watch the learned wind, and see why it samples in very **few steps**.
5. Use it on a **real physics problem** — sampling the conformations of **alanine dipeptide**, a
   benchmark molecule — and on tiny images.

> **Where this sits in the course.** This is the second stop in the generative-models block:
> **L15 normalizing flows / CNFs → L16 flow matching (← you are here) → L17 diffusion → capstone survey.**
> In L15 you built a continuous-time generative model (a CNF) and trained it by *maximum likelihood*,
> which needed an ODE solve and a divergence term at every training step. Flow matching gets the **same
> kind of model** (a velocity field that flows noise into data) but trains it by a plain **regression**
> — no ODE solve, no Jacobian, no architecture constraints. That is what "simulation-free" means.

> **Runtime.** Everything runs on a free Colab CPU in a few minutes. No GPU needed.
> **Math.** The main text has almost none. The proofs and the precise links to L15/L17 live in an
> **optional appendix** at the end.


## Setup — run this first

In [ ]:
# !pip install -q jax jaxlib optax flax matplotlib scikit-learn

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

KEY = jr.PRNGKey(42)   # course-wide default seed
print(f"JAX is running on: {jax.default_backend()}")


## 0. Where we left off (L15) — and the one thing we change

In **Lecture 15** you built a **normalizing flow** (RealNVP) and, at the end, its continuous-time
cousin, the **continuous normalizing flow (CNF)**. The job was always the same:

> learn an invertible recipe that turns **easy noise** $z \sim \mathcal{N}(0, I)$ into a **data
> sample** $x$, and vice-versa.

RealNVP did this by stacking *invertible coupling layers* and training them to **maximize the
likelihood** $\log p_\theta(x)$ — which forced us to (i) restrict the network to invertible blocks
and (ii) carry a Jacobian-determinant term. The CNF replaced the discrete stack with a continuous
*velocity field* $v_\theta(x,t)$ and a black-box ODE solver — elegant, but training it by likelihood
meant solving an ODE and computing a divergence $\nabla\!\cdot v_\theta$ **at every step**. Slow.

**Flow matching keeps the exact same end product** — a velocity field that flows noise into data —
**but throws away the expensive training objective.** Instead of maximizing likelihood, we will
*directly regress the velocity field against an arrow we already know*. No ODE solve during training,
no Jacobian, and **any** neural network will do. Same goal as L15; a dramatically cheaper road. Let's
build it from scratch.


## 1. What are we trying to do? (and why should you care?)

Here is the whole goal in one sentence:

> **Given a bunch of examples, build a machine that produces brand-new ones — similar to the examples, but not copies.**

To make it concrete, our "examples" today will be **dots arranged in the shape of a smiley face.** We want a machine that, after looking at these dots, can spit out *fresh* smiley faces — endless new ones, each a little different, none copied from the training set.

That sounds like a toy. It isn't — it is exactly how today's most powerful AI tools work:

- **🖼️ Image & video generators.** A photo is just a long list of numbers (the brightness of each pixel). "Realistic photos" form a strange, complicated blob inside that huge space of number-lists. A generator's job is to produce a *new* point inside that blob — i.e. a new realistic image. **Stable Diffusion 3, Flux, and Meta's Movie Gen are flow-matching models.** Our smiley is the same task in 2 dimensions instead of a million.
- **🧪 Molecules & materials.** A molecule constantly jiggles; the 3-D shapes it visits form a cloud. Generating new shapes from that cloud lets chemists explore a molecule's behavior, or invent new drugs and materials — *without* running weeks of physics simulation.
- **🌡️ Physics sampling.** Statistical physics constantly needs "typical states of this system at temperature $T$." Flow matching produces them directly (we'll do this on a real molecule, alanine dipeptide, in §10).

The common thread: **turn cheap randomness (noise) into valuable structure (a sample).** Anytime you can say *"here are some examples — give me more,"* flow matching is one of the best tools we currently have.

Let's look at our example data.


In [ ]:
# Our "data": ~8000 dots arranged as a smiley face. (Just a 2-D point cloud.)
def make_smiley(n, key):
    k = jr.split(key, 7)
    n_ring, n_eye, n_mouth = n // 2, n // 6, n // 4
    # face outline (a ring)
    th   = jr.uniform(k[0], (n_ring,)) * 2 * jnp.pi
    ring = jnp.stack([2.0*jnp.cos(th), 2.0*jnp.sin(th)], -1) + 0.05*jr.normal(k[1], (n_ring, 2))
    # two eyes (small blobs)
    eyeL = jnp.array([-0.8, 0.65]) + 0.10*jr.normal(k[2], (n_eye, 2))
    eyeR = jnp.array([ 0.8, 0.65]) + 0.10*jr.normal(k[3], (n_eye, 2))
    # smile (a lower arc)
    phi   = jnp.pi*1.15 + jr.uniform(k[4], (n_mouth,)) * jnp.pi*0.7
    smile = jnp.stack([1.2*jnp.cos(phi), 1.2*jnp.sin(phi)+0.15], -1) + 0.05*jr.normal(k[5], (n_mouth, 2))
    pts = jnp.concatenate([ring, eyeL, eyeR, smile], 0)
    return pts[jr.permutation(k[6], len(pts))]

data = make_smiley(8000, jr.PRNGKey(1))
noise = jr.normal(jr.PRNGKey(2), (8000, 2))   # what "random dots" look like

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].scatter(np.asarray(noise[:, 0]), np.asarray(noise[:, 1]), s=3, alpha=0.4, c="0.5")
axes[0].set_title("Random dots (noise) — our starting point")
axes[1].scatter(np.asarray(data[:, 0]),  np.asarray(data[:, 1]),  s=3, alpha=0.4, c="C0")
axes[1].set_title("Our data — the shape we want to reproduce")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()
print(f"data shape: {data.shape}   (8000 dots, each with an (x, y) position)")


**The challenge:** turn the gray cloud on the left into a fresh smiley like the one on the right — and be able to do it again and again, producing new smileys every time. Read on.


## 2. The big idea: a wind that carries dots into shape

Picture the 2-D plane with **wind blowing across it.** Drop a leaf anywhere and the wind carries it along some path. Now imagine we could *design* the wind so cleverly that **leaves dropped at completely random spots all drift until, together, they trace out our smiley face.** That designed wind *is* the generative model.

Two details make this precise:

- The wind is allowed to **change over time.** We use a "progress dial" $t$ that runs from $t=0$ (the beginning) to $t=1$ (the end). At $t=0$ the leaves are random dots; at $t=1$ they've formed the shape.
- The **neural network is the wind map.** You hand it a position and the current time, and it tells you which way the wind blows there and how strong:
$$
\underbrace{v_\theta(\text{position},\ t)}_{\text{the network}} \;=\; \text{the wind arrow at that place and time.}
$$
(The $\theta$ is just "the tunable knobs of the network." This $v_\theta(x,t)$ is exactly the **velocity field** of the L15 CNF — same object, different training.)

So **generating a sample = drop a random dot at $t=0$ and let the wind carry it to $t=1$.** Here is a cartoon of the idea — a made-up wind that herds dots toward two target blobs, with three leaf trajectories drawn in:


In [ ]:
# CARTOON (hand-made wind, for intuition only): wind points toward the nearest of two targets.
targets = np.array([[-1.0, 0.4], [1.0, 0.4]])
def wind(p):                       # p: (...,2) -> arrow (...,2)
    d = p[..., None, :] - targets  # vector from each target to p
    dist = np.linalg.norm(d, axis=-1, keepdims=True)
    nearest = np.argmin(dist[..., 0], axis=-1)
    pull = targets[nearest] - p    # head toward the nearest target
    return pull

gx, gy = np.meshgrid(np.linspace(-3, 3, 20), np.linspace(-3, 3, 20))
grid = np.stack([gx.ravel(), gy.ravel()], -1)
arr = wind(grid)
mag = np.hypot(arr[:, 0], arr[:, 1]) + 1e-9

fig, ax = plt.subplots(figsize=(6.2, 6))
ax.quiver(grid[:, 0], grid[:, 1], arr[:, 0]/mag, arr[:, 1]/mag, mag,
          cmap="Blues", scale=28, width=0.005, alpha=0.8)
# release 3 leaves from spread-out spots and let the wind carry them (simple Euler steps)
for c, start in zip(["C1", "C3", "C2"], [(-2.6, 2.4), (2.6, -2.2), (-0.3, -2.7)]):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(60):
        p = p + 0.04 * wind(p[None])[0]
        path.append(p.copy())
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], c=c, lw=2.2, zorder=4)
    ax.scatter(*path[0], c=c, s=70, marker="o", zorder=5, ec="k", lw=0.5)   # start (random)
ax.scatter(targets[:, 0], targets[:, 1], c="k", s=140, marker="X", zorder=6, label="targets")
ax.set_title("Cartoon: a wind field (arrows) carries leaves (colored paths)\nfrom random starts (•) toward the targets (✖)")
ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()


That wind was hand-made and silly (it just points at two blobs). The real question is: **how do we discover the wind that carries random dots into our actual smiley?** That's the clever part, and it's next.


## 3. How to discover the wind — the straight-line trick

We don't know the wind. But there's a cheat that hands us free training examples.

**Take one real data dot** (call it the *end*, since it's where we want to arrive at $t=1$).
**Take one random dot** (the *start*, at $t=0$).
Now *pretend* a leaf glides in a perfectly **straight line** from the start to the end as the dial $t$ goes $0 \to 1$. Its position at progress $t$ is just a blend of the two:

$$
\text{position}(t) \;=\; (1-t)\cdot\text{start} \;+\; t\cdot\text{end}.
$$

(At $t=0$ you get the start; at $t=1$ you get the end; in between, a straight blend.) And because it's a straight line at steady pace, we know its **arrow** (which way and how fast it's moving) at every instant — it's constant, pointing straight from start to end:

$$
\boxed{\;\text{arrow} \;=\; \text{end} - \text{start}.\;}
$$

That's a *free* training example: "at this position and this time, the wind should blow in this direction." Make **millions** of them (every pair of one random dot and one data dot gives one), and train the network to match the arrow. **That is the entire training algorithm.** Here are a few of those straight-line "belts":


In [ ]:
rng = np.random.default_rng(1)
starts = rng.normal(0, 1.0, size=(9, 2)) * np.array([0.5, 1.0]) + np.array([-2.2, 0.0])  # random
ends   = np.array([[2.0, 1.2], [2.1, -1.1], [1.8, 0.1], [2.2, 0.8], [1.9, -0.6],
                   [2.0, 0.5], [2.1, -0.2], [1.8, 1.0], [2.2, -0.9]])                     # "data"

fig, ax = plt.subplots(figsize=(8, 4.6))
ts = np.linspace(0, 1, 40)
for a, b in zip(starts, ends):
    ax.plot((1-ts)*a[0] + ts*b[0], (1-ts)*a[1] + ts*b[1], color="0.75", lw=1, zorder=1)
ax.scatter(starts[:, 0], starts[:, 1], c="0.5", s=70, zorder=3, label="start = random dot ($t=0$)")
ax.scatter(ends[:, 0],   ends[:, 1],   c="C0",  s=70, zorder=3, label="end = data dot ($t=1$)")
# one leaf mid-flight with its arrow
a, b = starts[2], ends[2]; tt = 0.45; p = (1-tt)*a + tt*b; arrow = b - a
ax.scatter(*p, c="k", s=90, zorder=4)
ax.annotate("", xy=p + 0.18*arrow, xytext=p, arrowprops=dict(arrowstyle="-|>", color="k", lw=2.2))
ax.text(p[0]-0.1, p[1]-0.5, "arrow = end − start", fontsize=11)
ax.set_title("The straight-line trick: each (random, data) pair gives one free training arrow")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(loc="upper left"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()


### ✏️ Your turn — code the three core pieces of flow matching

The whole method rests on three tiny functions. Before we hand you the finished training loop,
**you** will implement them. Each task has a function signature with a `TODO`, a quick self-check,
and — right below — a **collapsed reference implementation** (click *"Show code"* in Colab to reveal
it if you get stuck). Fill in the `TODO`s and run the self-checks until they print `OK`.

These three are *literally* the entire algorithm:

| | what it does | the formula |
|---|---|---|
| **A** `cfm_target_velocity` | the interpolated point **and** its true arrow | $x_t=(1-t)x_0+t\,x_1,\quad u_t=x_1-x_0$ |
| **B** `linear_interpolant` | just the point $x_t$ (with correct broadcasting) | $x_t=(1-t)x_0+t\,x_1$ |
| **C** `cfm_loss` | the regression loss we minimize | $\mathbb{E}\,\lVert v_\theta(x_t,t)-u_t\rVert^2$ |


#### Cell A — `cfm_target_velocity`

Given a batch of starts `x0`, ends `x1`, and per-sample times `t`, return the **interpolated
position** `x_t` and the **target arrow** `u_t`. This is the heart of conditional flow matching.


In [ ]:
def cfm_target_velocity(x0: jnp.ndarray, x1: jnp.ndarray, t: jnp.ndarray):
    # Conditional-flow-matching interpolant and target velocity.
    #   x0: (B, d) random start points  (t = 0)
    #   x1: (B, d) data end points      (t = 1)
    #   t:  (B,)   times in [0, 1], one per sample
    # Returns:
    #   x_t: (B, d) point on the straight line at progress t
    #   u_t: (B, d) the constant target arrow (end - start)
    # TODO: broadcast t from (B,) to (B, 1) so it multiplies each row of x0 / x1.
    # tb = ...
    # TODO: x_t = (1 - t) * x0 + t * x1   (the straight blend)
    # x_t = ...
    # TODO: u_t = x1 - x0                 (the constant arrow)
    # u_t = ...
    raise NotImplementedError("implement cfm_target_velocity, then delete this line")
    return x_t, u_t


# --- self-check (run after you fill in the TODOs) ---
_B, _d = 5, 2
_x0 = jr.normal(jr.PRNGKey(0), (_B, _d))
_x1 = jr.normal(jr.PRNGKey(1), (_B, _d))
_xt0, _ut = cfm_target_velocity(_x0, _x1, jnp.zeros(_B))   # t = 0  -> x_t == x0
_xt1, _   = cfm_target_velocity(_x0, _x1, jnp.ones(_B))    # t = 1  -> x_t == x1
assert _ut.shape == (_B, _d), f"u_t should be {(_B, _d)}, got {_ut.shape}"
assert jnp.allclose(_xt0, _x0), "at t=0, x_t must equal x0"
assert jnp.allclose(_xt1, _x1), "at t=1, x_t must equal x1"
print("OK  cfm_target_velocity: t=0 -> x0, t=1 -> x1, and u_t shape =", _ut.shape)


<details><summary>Reference implementation (Colab: click *Show code*)</summary>

In [ ]:
# @title Reference: cfm_target_velocity
def cfm_target_velocity(x0: jnp.ndarray, x1: jnp.ndarray, t: jnp.ndarray):
    tb  = t[:, None]                 # (B,) -> (B, 1) so it broadcasts over the d columns
    x_t = (1.0 - tb) * x0 + tb * x1  # straight blend
    u_t = x1 - x0                    # constant arrow, independent of t
    return x_t, u_t


</details>

#### Cell B — `linear_interpolant`

Same blend, but returning **only** the point. The point of this exercise is the **broadcasting**:
`t` comes in as shape `(B,)` and must be reshaped to `(B, 1)` before it multiplies the `(B, d)`
arrays — otherwise JAX will broadcast it the wrong way (or error).


In [ ]:
def linear_interpolant(x0: jnp.ndarray, x1: jnp.ndarray, t: jnp.ndarray) -> jnp.ndarray:
    # Return x_t = (1 - t) x0 + t x1 with correct (B,) -> (B,1) broadcasting.
    # TODO: reshape t to (B, 1), then return the straight blend.
    raise NotImplementedError("implement linear_interpolant, then delete this line")


# --- self-check ---
_x0 = jnp.array([[0.0, 0.0], [2.0, -2.0]])
_x1 = jnp.array([[4.0, 8.0], [0.0,  6.0]])
assert jnp.allclose(linear_interpolant(_x0, _x1, jnp.array([0.0, 0.0])), _x0)          # t=0
assert jnp.allclose(linear_interpolant(_x0, _x1, jnp.array([1.0, 1.0])), _x1)          # t=1
mid = linear_interpolant(_x0, _x1, jnp.array([0.5, 0.5]))
assert jnp.allclose(mid, jnp.array([[2.0, 4.0], [1.0, 2.0]])), f"midpoint wrong: {mid}"
print("OK  linear_interpolant: endpoints and midpoint all correct")


<details><summary>Reference implementation (Colab: click *Show code*)</summary>

In [ ]:
# @title Reference: linear_interpolant
def linear_interpolant(x0: jnp.ndarray, x1: jnp.ndarray, t: jnp.ndarray) -> jnp.ndarray:
    tb = t[:, None]                  # (B,) -> (B, 1)
    return (1.0 - tb) * x0 + tb * x1


</details>

#### Cell C — `cfm_loss`

Now wire it together into the **loss we minimize**. Sample random times $t$ and random starts $x_0$,
build the interpolant with your Cell-A function, ask the network for its arrow, and return the
**mean squared error** against the target arrow. Most of the scaffold is written — you only fill the
final MSE line. After it runs, we connect it to `jax.grad` / `nnx.value_and_grad` to confirm gradients
flow.


In [ ]:
def cfm_loss(net, x1_batch: jnp.ndarray, key) -> jnp.ndarray:
    # Conditional flow-matching loss for one batch of data points x1_batch (B, d).
    k1, k2 = jr.split(key)
    B = x1_batch.shape[0]
    t  = jr.uniform(k1, (B,))                       # random times in [0, 1]
    x0 = jr.normal(k2, x1_batch.shape)              # random starts ~ N(0, I)
    x_t, u_t = cfm_target_velocity(x0, x1_batch, t) # interpolant + target arrow (your Cell A)
    v_pred = net(x_t, t)                            # the network's guessed arrow
    # TODO: return the mean squared error between v_pred and u_t.
    # loss = jnp.mean(???)
    raise NotImplementedError("implement the MSE line in cfm_loss, then delete this line")
    return loss


<details><summary>Reference implementation (Colab: click *Show code*)</summary>

In [ ]:
# @title Reference: cfm_loss
def cfm_loss(net, x1_batch: jnp.ndarray, key) -> jnp.ndarray:
    k1, k2 = jr.split(key)
    B = x1_batch.shape[0]
    t  = jr.uniform(k1, (B,))
    x0 = jr.normal(k2, x1_batch.shape)
    x_t, u_t = cfm_target_velocity(x0, x1_batch, t)
    v_pred = net(x_t, t)
    return jnp.mean((v_pred - u_t) ** 2)            # MSE arrow regression


</details>

**Checkpoint.** Once all three self-checks print `OK`, the rest of the notebook just *uses* these
functions at scale. When we train below, watch the loss **drop and then flatten at a positive value**
— if it converges to a small positive number (not zero, not NaN), your `cfm_loss` is correct. (Why
positive, not zero? See §4.)


## 4. The only catch: the lines cross — so the network learns the *average*

Look again at the picture in §3: the straight-line belts **cross each other.** At a crossing point, two different leaves pass through heading in two different directions. The network is asked to output a *single* arrow there — it can't satisfy both at once.

So what does it do? When you train a network to match many conflicting targets at the same input, the best it can do (in the least-squares sense) is output their **average.** And here is the small miracle that makes flow matching work:

> **Following the averaged wind still carries every random dot into the correct shape.**

Let's *see* the averaging. Below (left) we draw lots of crossing belts in a simple 1-D example where the "data" is two spikes. Then (right) we compute, at each position, the **average arrow** of all belts passing through — just by bucketing positions and averaging. The tangle collapses into a clean, smooth wind:


In [ ]:
rng = np.random.default_rng(0)
N = 6000
x0 = rng.normal(0.0, 1.0, N)                                   # random starts (1-D)
x1 = np.where(rng.random(N) < 0.5, rng.normal(-2, 0.25, N),
                                    rng.normal( 2, 0.25, N))    # data = two spikes
vel = x1 - x0                                                  # each belt's arrow

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ts = np.linspace(0, 1, 30)
idx = rng.choice(N, 200, replace=False)
vn = (vel[idx] - vel[idx].min()) / (np.ptp(vel[idx]) + 1e-9)
for j, i in enumerate(idx):
    axes[0].plot(ts, (1-ts)*x0[i] + ts*x1[i], color=plt.cm.coolwarm(vn[j]), lw=0.7, alpha=0.7)
axes[0].set_title("200 crossing belts (color = each belt's arrow)")
axes[0].set_xlabel("progress $t$"); axes[0].set_ylabel("position"); axes[0].set_ylim(-4, 4)

for t in [0.2, 0.5, 0.8]:
    xt = (1-t)*x0 + t*x1
    bins = np.linspace(-4, 4, 41); which = np.digitize(xt, bins)
    cx, cy = [], []
    for b in range(1, len(bins)):
        m = which == b
        if m.sum() > 20:
            cx.append(0.5*(bins[b-1]+bins[b])); cy.append(vel[m].mean())
    axes[1].plot(cx, cy, "-o", ms=3, label=f"$t={t}$")
axes[1].axhline(0, color="k", lw=0.6)
axes[1].set_title("The average arrow at each position = the wind the network learns")
axes[1].set_xlabel("position"); axes[1].set_ylabel("average arrow"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


The right panel is a smooth, single-valued wind: **positive** (push right) where dots should head toward the $+2$ spike, **negative** (push left) toward the $-2$ spike, zero in the middle. Release the random cloud into this wind and it splits cleanly into the two spikes. *That averaged wind is exactly what our network will learn* — but smoothly, and in any number of dimensions.

(If you're wondering *why* following the average is guaranteed to land on the right shape — that's a real theorem, and it's in the optional appendix. For now, let's just build it and watch it work.)


## 5. Build the wind map (a small neural network)

The network takes a position $(x, y)$ and a time $t$, and returns a 2-D arrow — the wind there. It's a small "multilayer perceptron" (a stack of linear layers with bends in between). There is **no special structure required** — any ordinary network works. (Contrast L15's RealNVP, which had to be built out of invertible coupling blocks. Flow matching imposes no such constraint — that freedom is one of its biggest practical advantages.)

One small practical touch: instead of feeding the raw number $t$, we feed a handful of sine/cosine waves of $t$ (a "time embedding"). This just helps the network notice *when* it is — a standard trick, nothing deep. Read the code; it's short.


In [ ]:
def time_features(t, dim):
    # Turn a time t in [0,1] into `dim` sine/cosine features so the network can read the clock.
    half  = dim // 2
    freqs = jnp.exp(-jnp.log(10000.0) * jnp.arange(half) / max(half - 1, 1))
    ang   = (t * 1000.0)[:, None] * freqs[None, :]
    return jnp.concatenate([jnp.sin(ang), jnp.cos(ang)], axis=-1)


class WindMap(nnx.Module):
    # The wind map v_theta(position, t): given where & when, returns the wind arrow.
    def __init__(self, d, hidden=256, t_dim=64, *, rngs):
        self.d = d
        self.t_dim = t_dim
        self.t_net = nnx.Sequential(
            nnx.Linear(t_dim, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs),
        )
        self.net = nnx.Sequential(
            nnx.Linear(d + hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, d, rngs=rngs),
        )

    def __call__(self, x, t):
        # x: (B, d) positions,  t: (B,) times in [0,1].  Returns (B, d) wind arrows.
        tf = self.t_net(time_features(t, self.t_dim))
        return self.net(jnp.concatenate([x, tf], axis=-1))


# Quick check that shapes line up.
net = WindMap(d=2, rngs=nnx.Rngs(0))
xb = jr.normal(jr.PRNGKey(0), (32, 2)); tb = jr.uniform(jr.PRNGKey(1), (32,))
print(f"input: 32 positions + 32 times  ->  output arrows: {net(xb, tb).shape}")
print(f"tunable knobs (parameters): {sum(p.size for p in jax.tree.leaves(nnx.state(net, nnx.Param))):,}")


## 6. Train it — the whole recipe is six lines

In plain words, one training step is:

1. pick a random **time** $t$ between 0 and 1,
2. pick a **random start** dot and a **real data** dot,
3. place a leaf on the straight line between them at progress $t$,
4. ask the network for the wind arrow there,
5. nudge the network so its arrow is closer to (**end − start**),
6. repeat a few hundred thousand times.

No simulation, no calculus, no special tricks — just "guess the arrow, compare, fix." The loss
function below is **exactly the `cfm_loss` you wrote in Cell C** (inlined here so the training loop is
self-contained):


In [ ]:
def train(net, data, n_epochs=700, batch=256, lr=3e-4, seed=42, log_every=100):
    opt = nnx.Optimizer(net, optax.adam(lr), wrt=nnx.Param)

    def loss_fn(net, x1, key):                       # x1 = a batch of real data dots
        k1, k2 = jr.split(key)
        B   = x1.shape[0]
        t   = jr.uniform(k1, (B,))                    # 1. random times in [0,1]
        x0  = jr.normal(k2, x1.shape)                 # 2. random starts
        x_t = (1 - t)[:, None] * x0 + t[:, None] * x1 # 3. leaf on the straight line
        arrow_target = x1 - x0                        #    the true arrow (end - start)
        arrow_pred   = net(x_t, t)                    # 4. the network's guess
        return jnp.mean((arrow_pred - arrow_target) ** 2)   # 5. how wrong is the guess?

    @nnx.jit
    def step(net, opt, x1, key):
        loss, grads = nnx.value_and_grad(loss_fn, argnums=nnx.DiffState(0, nnx.Param))(net, x1, key)
        opt.update(net, grads)
        return loss

    losses, key = [], jr.PRNGKey(seed)
    for epoch in range(n_epochs):
        key, kp = jr.split(key)
        shuffled = data[jr.permutation(kp, len(data))]
        ep, nb = 0.0, 0
        for i in range(0, len(data), batch):
            xb = shuffled[i:i+batch]
            if len(xb) < 2:
                continue
            key, ks = jr.split(key)
            ep += float(step(net, opt, xb, ks)); nb += 1
        losses.append(ep / max(nb, 1))
        if (epoch + 1) % log_every == 0:
            print(f"epoch {epoch+1:4d}   loss = {losses[-1]:.4f}")
    return losses


In [ ]:
net = WindMap(d=2, rngs=nnx.Rngs(0))
history = train(net, data, n_epochs=700)        # ~1-2 minutes on a CPU

plt.figure(figsize=(7, 3))
plt.plot(history); plt.xlabel("epoch"); plt.ylabel("loss (how wrong the arrows are)")
plt.title("Training the wind map on smiley data"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


**The loss settles at a positive value — not zero — and that's correct, not a bug.** Remember the crossing belts: at a given spot the true arrows genuinely disagree, so the best possible guess is their average, and the leftover disagreement keeps the loss above zero. A loss that *drops then flattens* is exactly what success looks like here. (This is the checkpoint promised after Cell C.)


## 7. Generate new smileys — just follow the wind

Now the fun part. To make a new sample: drop a **random dot** at $t=0$ and **step it along the wind** until $t=1$. Each step nudges the dot a little in the direction the wind points:

$$
\text{new position} \;=\; \text{position} \;+\; (\text{step size}) \times v_\theta(\text{position},\ t).
$$

That's it — repeat for a few dozen steps from $t=0$ to $t=1$. (We use a slightly smarter "midpoint" version of the step that's more accurate, but the idea is identical. This is exactly the ODE solve from the L15 CNF — only now the velocity field was trained by regression, not likelihood.)


In [ ]:
@nnx.jit
def wind_step(net, x, t, dt):
    # midpoint step: look ahead half a step, then move using the wind there (more accurate).
    B  = x.shape[0]
    a1 = net(x, jnp.full((B,), t))
    a2 = net(x + 0.5 * dt * a1, jnp.full((B,), t + 0.5 * dt))
    return x + dt * a2

def generate(net, n, key, n_steps=100, save_every=None):
    x  = jr.normal(key, (n, net.d))           # start: random dots at t=0
    dt = 1.0 / n_steps
    snaps = {0.0: np.asarray(x)} if save_every else None
    for i in range(n_steps):                  # follow the wind from t=0 to t=1
        x = wind_step(net, x, jnp.float32(i * dt), jnp.float32(dt))
        if save_every and ((i + 1) % save_every == 0 or i == n_steps - 1):
            snaps[round((i + 1) * dt, 2)] = np.asarray(x)
    return x, snaps


In [ ]:
# Watch random dots flow into a smiley, step by step.
final, snaps = generate(net, 5000, jr.PRNGKey(0), n_steps=100, save_every=20)
times = sorted(snaps.keys())
fig, axes = plt.subplots(1, len(times), figsize=(2.5*len(times), 2.8))
for ax, t in zip(axes, times):
    p = snaps[t]
    ax.scatter(p[:, 0], p[:, 1], s=2, alpha=0.4, c="C1")
    ax.set_title(f"$t = {t:.1f}$"); ax.set_aspect("equal")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(alpha=0.2); ax.tick_params(labelsize=7)
plt.suptitle("Random dots ($t=0$) follow the learned wind and become a smiley ($t=1$)", y=1.05)
plt.tight_layout(); plt.show()


In [ ]:
# Side by side: real data vs freshly generated.
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
axes[0].scatter(np.asarray(data[:, 0]),  np.asarray(data[:, 1]),  s=3, alpha=0.4, c="C0")
axes[0].set_title("Real data")
axes[1].scatter(np.asarray(final[:, 0]), np.asarray(final[:, 1]), s=3, alpha=0.4, c="C1")
axes[1].set_title(f"Generated by our model ({len(final)} new dots)")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


🎉 **It made a brand-new smiley** — two eyes, a smile, a face — out of nothing but random dots and a learned wind. None of these dots is a copy of a training dot; they're freshly produced. The same six-line recipe, pointed at photos or molecules instead of smileys, is what powers real generative AI.


## 8. Look at the wind it learned

Since the network *is* the wind, we can simply draw it. Below are arrows of $v_\theta$ at several times (color = wind strength), with the data faintly overlaid. Early on the wind sweeps dots broadly toward where the face will be; later it gently parks them onto the eyes, smile, and ring.


In [ ]:
def wind_grid(net, t, lim=3, n=24):
    g = jnp.linspace(-lim, lim, n)
    xx, yy = jnp.meshgrid(g, g)
    pts = jnp.stack([xx.ravel(), yy.ravel()], -1)
    v = net(pts, jnp.full((pts.shape[0],), float(t)))
    return np.asarray(xx), np.asarray(yy), np.asarray(v)

times = [0.0, 0.3, 0.6, 0.9]
fig, axes = plt.subplots(1, len(times), figsize=(3.2*len(times), 3.3))
for ax, t in zip(axes, times):
    xx, yy, v = wind_grid(net, t)
    vx = v[:, 0].reshape(xx.shape); vy = v[:, 1].reshape(xx.shape)
    mag = np.hypot(vx, vy) + 1e-9
    ax.quiver(xx, yy, vx/mag, vy/mag, mag, scale=26, cmap="viridis", width=0.006)
    ax.scatter(np.asarray(data[:400, 0]), np.asarray(data[:400, 1]), s=3, alpha=0.4, c="orange")
    ax.set_title(f"wind at $t = {t}$"); ax.set_aspect("equal")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.tick_params(labelsize=7)
plt.suptitle("The learned wind field at different times (color = strength)", y=1.04)
plt.tight_layout(); plt.show()


## 9. Why it's *fast*: a few steps are enough

Because every training belt was a **straight line**, the wind tends to push dots along nearly-straight paths — and straight paths are easy to follow with big steps. Watch what happens if we take only a handful of steps from $t=0$ to $t=1$:


In [ ]:
@nnx.jit
def euler_step(net, x, t, dt):
    return x + dt * net(x, jnp.full((x.shape[0],), t))

def generate_euler(net, n, key, n_steps):
    x = jr.normal(key, (n, net.d)); dt = 1.0 / n_steps
    for i in range(n_steps):
        x = euler_step(net, x, jnp.float32(i * dt), jnp.float32(dt))
    return np.asarray(x)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, N in zip(axes, [2, 4, 8, 50]):
    p = generate_euler(net, 5000, jr.PRNGKey(0), N)
    ax.scatter(p[:, 0], p[:, 1], s=2, alpha=0.4, c="C1")
    ax.set_title(f"{N} step{'s' if N>1 else ''}")
    ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.grid(alpha=0.2)
plt.suptitle("Straight paths → even a handful of steps already gives a smiley", y=1.04)
plt.tight_layout(); plt.show()


Even 4–8 steps already look like a smiley. This is a big practical deal: the slower generative models that came before (diffusion models) often needed *hundreds* of steps. Flow matching's straight paths are a major reason today's image and video generators are fast — and with an extra "straightening" trick (called *reflow*), people push this down to a **single step.**

> **→ Looking ahead to L17 (Diffusion).** Diffusion models are the close cousin you'll meet next. They
> are *also* a velocity/CNF in disguise (their "probability-flow ODE" is exactly a continuous
> normalizing flow), but they ride a **curved, stochastic path** from noise to data instead of our
> straight line. The curve is longer and wigglier, which is *why* diffusion typically needs many more
> integration steps than flow matching. In exchange, diffusion learns the **score** $\nabla\log p_t$,
> which gives an *exact* reverse process and a principled way to add noise back. We'll see in L17 that
> straight-path flow matching and curved-path diffusion are two members of the **same family** — and
> the appendix (A5) below makes that precise.


## 10. A physics payoff: a Boltzmann generator for a real molecule

Here's a problem physicists and chemists face constantly. A physical system prefers **low-energy**
states. If the system has energy $U(x)$ at temperature $T$, the probability of finding it at
configuration $x$ is the **Boltzmann distribution**

$$
p(x) \;\propto\; e^{-U(x)/k_BT}.
$$

We very often need **independent samples** from this distribution — to compute average properties,
reaction rates, free energies, and so on. The standard tool, molecular dynamics or Monte Carlo, takes
small local steps and gets **stuck** in whichever low-energy basin it starts in: crossing the barrier
between basins is rare and slow. A **Boltzmann generator** (Noé et al., *Science* 2019) instead trains
a generative model to produce equilibrium states *in one shot*, and then **reweights** them to be
exactly correct. Flow matching is a natural fit.

### Our molecule: alanine dipeptide

**Alanine dipeptide** (Ac-Ala-NMe) is the *Drosophila* of biomolecular simulation — the smallest
peptide that already shows the essential physics of protein folding. Its slow conformational change is
captured by just **two backbone dihedral angles**, $\phi$ and $\psi$. Plotting samples in the
$(\phi,\psi)$ plane gives the famous **Ramachandran plot**, whose populated regions are the molecule's
metastable states:

- the **$\alpha_R$ (right-handed $\alpha$-helix)** basin near $(\phi,\psi)\approx(-60^\circ,-45^\circ)$,
- the **$\beta$ / PPII (extended sheet)** basin near $(\phi,\psi)\approx(-120^\circ,+130^\circ)$,
- a sparsely populated **$\alpha_L$ (left-handed)** basin near $(+60^\circ,+40^\circ)$.

A correct sampler must populate **both** major basins in the right proportion — exactly the multi-modal
challenge local samplers fail at. We'll load ~2000 pre-computed $(\phi,\psi)$ frames (extracted from a
short AMBER99SB / 300 K simulation), train the **same `WindMap`** on them, generate, and check the
basins.


### Load the alanine dipeptide data

We host the pre-extracted angles as a small `ala2_ramachandran.npz` (shape `(~2000, 2)`, columns
$(\phi,\psi)$ in **radians**) on Google Drive so the notebook stays light and reproducible — no
OpenMM / MD run needed in Colab. The cell tries the download first; if the network is unavailable it
falls back to a faithful **synthetic** Ramachandran sampler built from a coarse AMBER-like free-energy
surface, so the rest of the notebook always runs.

> *How the real file was made (for reference):* a 300 K NVT simulation of Ac-Ala-NMe in AMBER99SB
> (via OpenMM, as packaged by `bgmol`/`bgflow`), saving $(\phi,\psi)$ every few ps for ~2000 frames.
> The synthetic fallback below reproduces the same basin structure so the physics lesson is identical.


In [ ]:
# --- Load alanine dipeptide (phi, psi) angles in radians, shape (~2000, 2). ---
# Primary path: download the pre-extracted npz from Google Drive.
# Fallback: synthesize from a coarse AMBER-like Ramachandran free-energy surface.

DRIVE_FILE_ID = "REPLACE_WITH_DRIVE_FILE_ID"   # ala2_ramachandran.npz on the course Drive
npz_path = "ala2_ramachandran.npz"

def _try_download():
    import os
    if os.path.exists(npz_path):
        return True
    try:
        import urllib.request
        url = f"https://drive.google.com/uc?export=download&id={DRIVE_FILE_ID}"
        urllib.request.urlretrieve(url, npz_path)
        # sanity: a real npz starts with the PK zip magic
        with open(npz_path, "rb") as f:
            return f.read(2) == b"PK"
    except Exception as e:
        print(f"(download unavailable: {e}) -- using synthetic fallback")
        return False

def _ramachandran_logp(phi, psi):
    # Coarse AMBER-like dimensionless free energy F(phi,psi); logp = -F (up to a constant).
    # Three Gaussian wells at alpha_R, beta/PPII, alpha_L with realistic relative depths.
    wells = [  # (phi0_deg, psi0_deg, sphi_deg, spsi_deg, depth)
        (-63.0,  -43.0, 18.0, 20.0, 1.00),   # alpha_R  (deepest, most populated)
        (-120.0, 130.0, 22.0, 24.0, 0.85),   # beta / PPII
        ( 60.0,   40.0, 16.0, 18.0, 0.35),   # alpha_L  (shallow, rarely visited)
    ]
    d2r = jnp.pi / 180.0
    acc = jnp.zeros_like(phi)
    for p0, q0, sp, sq, w in wells:
        # periodic angular distance so the surface wraps correctly on the torus
        dphi = jnp.arctan2(jnp.sin(phi - p0*d2r), jnp.cos(phi - p0*d2r))
        dpsi = jnp.arctan2(jnp.sin(psi - q0*d2r), jnp.cos(psi - q0*d2r))
        acc = acc + w * jnp.exp(-0.5*((dphi/(sp*d2r))**2 + (dpsi/(sq*d2r))**2))
    return jnp.log(acc + 1e-9)   # logp ∝ -F

def _synthesize(n, key):
    # Grid-based exact sampling from the coarse surface (same trick as the toy double-well had).
    g = jnp.linspace(-jnp.pi, jnp.pi, 240)
    gx, gy = jnp.meshgrid(g, g, indexing="xy")
    grid = jnp.stack([gx.ravel(), gy.ravel()], -1)
    logp = _ramachandran_logp(grid[:, 0], grid[:, 1]); logp -= logp.max()
    p = jnp.exp(logp); p /= p.sum()
    k1, k2 = jr.split(key)
    idx = jr.choice(k1, p.shape[0], (n,), p=p)
    dphi = 2*jnp.pi/(240-1)
    return grid[idx] + jr.uniform(k2, (n, 2), minval=-dphi/2, maxval=dphi/2)

if _try_download():
    angles = jnp.asarray(np.load(npz_path)["angles"], dtype=jnp.float32)   # (N, 2) radians
    print(f"loaded real ala2 angles: {angles.shape}")
else:
    angles = _synthesize(2000, jr.PRNGKey(0)).astype(jnp.float32)
    print(f"synthetic ala2 angles:  {angles.shape}")

data_phys = angles                                   # (N, 2) = (phi, psi) in radians
print(f"data_phys shape: {data_phys.shape}   (phi, psi in radians)")
print(f"phi range [deg]: [{float(jnp.degrees(data_phys[:,0].min())):.0f}, {float(jnp.degrees(data_phys[:,0].max())):.0f}]")
print(f"psi range [deg]: [{float(jnp.degrees(data_phys[:,1].min())):.0f}, {float(jnp.degrees(data_phys[:,1].max())):.0f}]")


In [ ]:
# The AMBER-like free-energy surface and the reference samples on it.
# (logp = -F up to a constant; this is the ground truth we'll reweight against.)
def energy_U(x):
    # "energy" = free energy F(phi, psi) = -logp, in dimensionless (kT) units.
    return -_ramachandran_logp(x[..., 0], x[..., 1])

kT = 1.0   # F is already in kT units

g = jnp.linspace(-jnp.pi, jnp.pi, 220); xx, yy = jnp.meshgrid(g, g)
U_grid = np.asarray(energy_U(jnp.stack([xx, yy], -1)))
deg = np.degrees

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
im = axes[0].contourf(deg(np.asarray(xx)), deg(np.asarray(yy)), U_grid, levels=25, cmap="magma")
axes[0].set_title("AMBER-like free energy $F(\\phi,\\psi)$  (lower = preferred)")
plt.colorbar(im, ax=axes[0], label="$F$ [$k_BT$]")
axes[1].scatter(deg(np.asarray(data_phys[:,0])), deg(np.asarray(data_phys[:,1])),
                s=5, alpha=0.35, c="C2")
axes[1].set_title(f"{data_phys.shape[0]} reference frames (Ramachandran plot)")
for ax in axes:
    ax.set_xlabel("$\\phi$ [deg]"); ax.set_ylabel("$\\psi$ [deg]")
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180); ax.set_aspect("equal")
    ax.axhline(0, color="w", lw=0.4, alpha=0.4); ax.axvline(0, color="w", lw=0.4, alpha=0.4)
plt.tight_layout(); plt.show()


**The usual way to sample this is slow.** A Monte Carlo / MD walk that takes small local steps gets **stuck** in whichever basin it starts in — crossing the $\alpha_R \leftrightarrow \beta$ barrier is a rare event. **Flow matching gives independent samples instantly:** train the wind on example states, then generate. Because every generated dot starts from a *random* Gaussian cloud, it lands in either basin according to the learned weights — no barrier to cross. Let's train and compare. *(We reuse the identical `train`, `generate`, `wind_grid`, and `log_prob` code — only the data changed.)*


In [ ]:
net_phys = WindMap(d=2, rngs=nnx.Rngs(11))
_ = train(net_phys, data_phys, n_epochs=300, log_every=150)   # ~1 min on CPU
gen_phys, _ = generate(net_phys, 4000, jr.PRNGKey(99), n_steps=100)
print(f"generated {gen_phys.shape[0]} samples")


In [ ]:
# A local Monte Carlo walk on the free-energy surface (it will get trapped in one basin).
@partial(jax.jit, static_argnames=("n_steps",))
def monte_carlo(x0, kT, n_steps, step, key):
    def one(carry, k):
        x, _ = carry
        k1, k2 = jr.split(k)
        prop = x + step * jr.normal(k1, x.shape)
        # keep angles in (-pi, pi]
        prop = (prop + jnp.pi) % (2*jnp.pi) - jnp.pi
        dE = (energy_U(prop) - energy_U(x)) / kT
        acc = jr.uniform(k2) < jnp.exp(-dE)
        x = jnp.where(acc, prop, x)
        return (x, 0), x
    _, traj = jax.lax.scan(one, (x0, 0), jr.split(key, n_steps))
    return traj

alphaR = jnp.radians(jnp.array([-63.0, -43.0]))    # start in the alpha-helix basin
beta   = jnp.radians(jnp.array([-120.0, 130.0]))   # start in the beta/PPII basin
walk_A = monte_carlo(alphaR, kT, 8000, 0.08, jr.PRNGKey(0))
walk_B = monte_carlo(beta,   kT, 8000, 0.08, jr.PRNGKey(1))

deg = np.degrees
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
axes[0].scatter(deg(np.asarray(walk_A)[:,0]), deg(np.asarray(walk_A)[:,1]), s=2, alpha=0.2, c="C3")
axes[0].set_title("Monte Carlo — started in $\\alpha_R$ (stuck)")
axes[1].scatter(deg(np.asarray(walk_B)[:,0]), deg(np.asarray(walk_B)[:,1]), s=2, alpha=0.2, c="C4")
axes[1].set_title("Monte Carlo — started in $\\beta$ (stuck)")
axes[2].scatter(deg(np.asarray(gen_phys)[:,0]), deg(np.asarray(gen_phys)[:,1]), s=2, alpha=0.25, c="C1")
axes[2].set_title("Flow matching — both basins, in one shot")
for ax in axes:
    ax.set_xlabel("$\\phi$ [deg]"); ax.set_ylabel("$\\psi$ [deg]")
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180); ax.set_aspect("equal"); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

# Physics checkpoint: did the generator populate BOTH major basins?
phi_deg = deg(np.asarray(gen_phys[:, 0])); psi_deg = deg(np.asarray(gen_phys[:, 1]))
in_alphaR = np.mean((phi_deg < 0) & (psi_deg < 30))
in_beta   = np.mean((phi_deg < 0) & (psi_deg > 60))
print(f"generated fraction near alpha_R basin (phi<0, psi<30):  {100*in_alphaR:.0f}%")
print(f"generated fraction near beta/PPII basin (phi<0, psi>60): {100*in_beta:.0f}%")
print("CHECKPOINT: both should be clearly > 0 -- the generator is NOT trapped in one basin.")


**Physics checkpoint.** The Monte Carlo walks each stay trapped in the basin they started in; the
flow-matching samples fill **both** the $\alpha_R$ and $\beta$ basins, **independently**, with no
barrier to cross. That multi-modal coverage in a single shot is the whole reason **Boltzmann
generators** are exciting for molecular physics.

### Bonus: getting the physics *exactly* right (reweighting)

Our model is only approximate. But it has a quietly powerful feature: for any configuration it
produces, it can also report **how likely it was to produce that configuration** ($p_\theta(x)$).
Knowing both the *true* desired probability $e^{-U/k_BT}$ and the model's $p_\theta$, we attach a
correcting **weight** to each sample,

$$
w(x) \;\propto\; \frac{e^{-U(x)/k_BT}}{p_\theta(x)},
$$

and weighted averages become **exactly correct for the true distribution**, even though the model
isn't perfect. Here $U$ is the **AMBER-like free energy** $F(\phi,\psi)$ from above — the same force
field that generated the data, so this is a genuine apples-to-apples physics check. (The model's
probability comes from tracking how the wind squeezes or spreads the cloud as dots flow — exactly the
**instantaneous change-of-variables** from L15, integrated along the ODE; see appendix A4.)


In [ ]:
# Model log-probability via the instantaneous change-of-variables (the L15 CNF formula),
# integrating the wind's divergence along the flow run backward from t=1 to t=0.
@nnx.jit
def _back_step(net, x, t, dt):
    B = x.shape[0]
    v = net(x, jnp.full((B,), t))
    def spread_rate(xi, ti):                 # local divergence: trace of the wind's Jacobian
        J = jax.jacfwd(lambda z: net(z[None], ti[None])[0])(xi)
        return jnp.trace(J)
    rate = jax.vmap(spread_rate)(x, jnp.full((B,), t))
    return x - dt * v, rate * dt

def log_prob(net, x_data, n_steps=100):
    B, d = x_data.shape
    dt = 1.0 / n_steps
    x, total = x_data, jnp.zeros((B,))
    for i in range(n_steps):
        x, dr = _back_step(net, x, jnp.float32(1.0 - i*dt), jnp.float32(dt))
        total = total + dr
    logp0 = -0.5*jnp.sum(x**2, -1) - 0.5*d*jnp.log(2*jnp.pi)   # N(0,I) start density
    return logp0 - total

# Average free energy <F> three ways: naive model, reweighted, and the grid-exact answer.
samp, _ = generate(net_phys, 3000, jr.PRNGKey(7), n_steps=100)
logq = log_prob(net_phys, samp, n_steps=100)
logw = -energy_U(samp)/kT - logq; logw -= jnp.max(logw)
w = jnp.exp(logw); w /= jnp.sum(w)
ess = float(1.0/jnp.sum(w**2)/samp.shape[0])

gf = jnp.linspace(-jnp.pi, jnp.pi, 360); fxx, fyy = jnp.meshgrid(gf, gf)
fpts = jnp.stack([fxx.ravel(), fyy.ravel()], -1)
pe = jnp.exp(-energy_U(fpts)/kT - jnp.max(-energy_U(fpts)/kT)); pe /= jnp.sum(pe)
exact_U = float(jnp.sum(pe * energy_U(fpts)))

print(f"effective sample size: {100*ess:.0f}%  (higher = better model)\n")
print(f"average free energy <F> [k_BT]:")
print(f"   model, no correction : {float(jnp.mean(energy_U(samp))):.3f}")
print(f"   model, reweighted    : {float(jnp.sum(w*energy_U(samp))):.3f}")
print(f"   grid-exact answer    : {exact_U:.3f}")


The reweighted estimate lands essentially on the grid-exact answer. **Approximate generator + a
correcting weight = exact physics.** This is precisely the workflow behind flow-matching Boltzmann
generators for real molecules (alanine dipeptide, Lennard-Jones clusters), where reweighted free
energies match slow gold-standard simulations.

> **→ Looking ahead to L19 (Simulation-Based Inference).** Everything here was *unconditional*: one
> fixed target $p(x)\propto e^{-U/k_BT}$. If instead you make the flow **conditional** — feed an
> observation $y$ into the wind map, $v_\theta(x,t\,|\,y)$ — the very same machinery learns a family of
> distributions $p_\theta(\theta\,|\,y)$. Trained on simulated $(\theta, y)$ pairs, that *is* a
> **neural posterior estimator**: a generative model of the parameters consistent with data. In L19
> we'll use exactly this to solve **inverse problems** in physics, where the likelihood is intractable
> but you can simulate forward.


## 11. It works on real images too (tiny digits)

An image is just more numbers, so the *same code* generates images. We use scikit-learn's 8×8 handwritten digits, squeezed to 16 numbers each (via PCA, a standard compression), and train the identical wind map — only the dimension changes from 2 to 16.


In [ ]:
try:
    import sklearn  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = digits.data.astype("float32")
scaler = StandardScaler().fit(X)
pca = PCA(n_components=16, whiten=True, random_state=0).fit(scaler.transform(X))
X16 = pca.transform(scaler.transform(X)).astype("float32")

fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img, lab in zip(axes.flat, X[:16], digits.target[:16]):
    ax.imshow(img.reshape(8, 8), cmap="gray_r"); ax.axis("off"); ax.set_title(str(lab), fontsize=9)
plt.suptitle("Real handwritten digits (our image data)", y=1.02); plt.tight_layout(); plt.show()


In [ ]:
net_digits = WindMap(d=16, rngs=nnx.Rngs(0))
_ = train(net_digits, jnp.asarray(X16), n_epochs=400, batch=128, log_every=200)

gen16, _ = generate(net_digits, 16, jr.PRNGKey(7), n_steps=100)   # generate 16 new "digits"
imgs = scaler.inverse_transform(pca.inverse_transform(np.asarray(gen16))).reshape(-1, 8, 8)
fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img in zip(axes.flat, imgs):
    ax.imshow(np.clip(img, 0, 16), cmap="gray_r"); ax.axis("off")
plt.suptitle("Brand-new digit images made by the same wind-map code", y=1.02); plt.tight_layout(); plt.show()


You should see digit-like glyphs — loops, strokes, slanted bars — generated by the *exact same six-line recipe* you used for smileys. For sharp, full-resolution images you'd swap the small network for a bigger one (a U-Net) and train on real photos; **the recipe doesn't change.**


## 12. Where flow matching is used in the real world

- **🖼️ Images & video.** Stable Diffusion 3, Flux, Meta Movie Gen — the wind blows random static into a photo or a video frame.
- **🧬 Proteins & drugs.** Models like FrameFlow and FoldFlow generate new 3-D protein backbones; flow matching replaced slower methods because of its few-step generation.
- **💎 Materials.** FlowMM generates new crystal structures (respecting the repeating lattice), proposing candidate materials.
- **🧪 Molecular sampling / free energy.** "Boltzmann generators" (our §10, scaled up) produce equilibrium molecular states and, via reweighting, exact free energies — far faster than brute-force simulation.
- **⚛️ Lattice field theory.** The same idea samples quantum-field configurations from $e^{-S}$ (the action $S$ plays the role of the energy), helping with calculations that ordinary simulation finds painfully slow.

The pattern is always the same: **you have examples (or an energy that defines them), and you want fast, independent new samples.** That's the flow-matching sweet spot.


## 13. Recap — the whole method on one page

**The picture:** a generative model is a **wind** (a velocity field $v_\theta(x,t)$) that carries random dots into a target shape; the neural network *is* the wind.

**Training (learn the wind):**
1. Blend a random *start* and a real *end*: $\;x_t = (1-t)\,x_0 + t\,x_1$.
2. The true arrow is $\;x_1 - x_0$ (end minus start).
3. Train the network so its arrow at $(x_t, t)$ matches it: minimize $\;\|v_\theta(x_t,t) - (x_1-x_0)\|^2$.

(The belts cross, so the network learns the *average* arrow — and that average is exactly the wind that reproduces the shape.)

**Generating (follow the wind):** drop a random dot at $t=0$ and step it along $v_\theta$ to $t=1$. A few steps suffice because the paths are nearly straight.

**Where it sits:** same end product as the **L15** CNF (a velocity field flowing noise→data), but trained by cheap regression instead of likelihood; the curved-path **L17** diffusion model is its close cousin; making the flow *conditional* turns it into the posterior estimator of **L19**.

**Why it's great:** dead-simple training (plain "match the arrow"), any network architecture, fast few-step generation, independent samples, and — for physics — an exact-reweighting bonus. It's the engine behind today's leading image, video, molecule, and material generators.

> Want the math (why averaging is provably correct, the continuity equation, I-CFM vs OT-CFM, the precise link to L15 and L17)? It's all in the optional appendix below.


## Appendix (optional) — the math behind the wind

Everything above was deliberately math-light. Here is the rigorous version, for the curious. None of it is needed to *use* flow matching.

### A1. The wind and the density: the continuity equation
Let $p_t(x)$ be the density of our cloud of dots at time $t$, and $v_t(x)$ the wind. As dots ride the wind, probability mass is neither created nor destroyed, so the density obeys the **continuity (transport) equation** — the same conservation law as for mass or charge in physics:
$$
\frac{\partial p_t(x)}{\partial t} + \nabla\!\cdot\!\big(p_t(x)\,v_t(x)\big) = 0 .
$$
The pair $(p_t, v_t)$ is consistent exactly when this holds. Our goal: find a wind $v_t$ that drives $p_0=\mathcal{N}(0,I)$ (random dots) to $p_1=$ data.

### A2. Why averaging the arrows is exactly right (Conditional Flow Matching)
We never know the ideal "marginal" wind $u_t(x)$ that does the job. But we *do* know the arrow of each single straight-line belt: for a belt from $x_0$ to $x_1$, it is $u_t(x\mid x_1)=x_1-x_0$. The ideal wind turns out to be the **average of those belt-arrows over all belts passing through $x$:**
$$
u_t(x) = \mathbb{E}\big[\,x_1 - x_0 \;\big|\; x_t = x\,\big].
$$
Training a network with the loss $\mathbb{E}\,\|v_\theta(x_t,t) - (x_1-x_0)\|^2$ has precisely this conditional average as its minimizer (regressing on a noisy target recovers its conditional mean). Formally, this "conditional" loss has the **same gradient** as the impossible "ideal" loss $\mathbb{E}\,\|v_\theta - u_t\|^2$:
$$
\nabla_\theta\,\mathcal{L}_{\text{conditional}}(\theta) = \nabla_\theta\,\mathcal{L}_{\text{ideal}}(\theta).
$$
So minimizing the easy, samplable loss is equivalent to minimizing the one we actually want. *(Proof: expand both squared norms; the cross-terms match after using the averaging identity, and the leftover terms don't depend on $\theta$.)* This is the **Conditional Flow Matching theorem** (Lipman et al. 2023; Tong et al. 2024).

### A3. Is the straight line the optimal-transport path? (I-CFM vs OT-CFM)
A common misconception is that the straight interpolant $x_t=(1-t)x_0+t\,x_1$ already *is* the
optimal-transport (OT) path. **It is not.** The subtlety is the difference between a *single* belt and
the *marginal* wind the network actually learns.

- **I-CFM (independent coupling), what we did above.** We pair each data point $x_1$ with an
  **independently** drawn noise $x_0\sim\mathcal{N}(0,I)$. *Each individual* belt is a straight line —
  but different pairs' straight lines **cross**. By §4/A2 the network learns the *average* arrow at each
  point, so the resulting **marginal** flow that transports $p_0\to p_1$ is **curved**, not straight,
  and is generally **not** the OT map.
- **OT-CFM (optimal-transport coupling).** Within each minibatch, first solve the discrete OT
  assignment between the noise batch and the data batch (cost $C_{ij}=\lVert x_0^i-x_1^j\rVert^2$, via
  an exact LP or Sinkhorn), **reorder** the pairs, then train with the *same* loss. The interpolant
  formula is identical,
  $$x_t=(1-t)\,x_0+t\,x_1\qquad(\text{both I-CFM and OT-CFM}),$$
  but the **coupling** differs: I-CFM draws $(x_0,x_1)$ **independently**, while OT-CFM draws them from a
  coupling that (approximately) **minimizes the Wasserstein-2 transport cost**. This OT coupling makes
  the *marginal* paths cross far less, so the learned marginal flow is genuinely **straighter** — which
  is what enables reliable **few-/one-step** generation. (As the interpolation noise $\sigma\to 0$,
  OT-CFM recovers the dynamic-OT solution; Tong et al. 2024, Prop. 3.4.) Iterating the straightening
  (re-pairing noise to its own generated output and retraining) is **rectified flow / reflow**, which
  drives toward a single-step map (Liu et al. 2023).

**Takeaway:** straight *belts* $\neq$ straight *marginal flow*. I-CFM is the simplest member of the
family; OT-CFM/reflow are the knobs that actually straighten the transport.

### A4. The model's probability (used for reweighting)
Following a dot along the wind, its log-probability changes at a rate set by how the wind locally spreads or squeezes the cloud (the divergence $\nabla\!\cdot v$) — this is the **instantaneous change-of-variables** introduced in L15 for CNFs:
$$
\log p_\theta(x_1) = \log p_0(x_0) - \int_0^1 \nabla\!\cdot v_\theta\big(x(t),t\big)\,dt ,
$$
where $x_0$ is found by running the wind **backward** from $x_1$. That integral is what the `log_prob` code accumulates, and it's why we can reweight to get exact physics. Note that, unlike L15's RealNVP, we **never needed** this divergence during *training* — only at evaluation time, when we want a likelihood.

### A5. Relationship to other generative models (L15 normalizing flows, L17 diffusion)
It is worth being precise about how flow matching relates to the models in the neighboring lectures,
because the difference is in the **training objective**, not the end product.

- **Normalizing flows / RealNVP (L15).** A normalizing flow learns a **single, fixed,
  time-independent invertible map** $f_\theta: z \mapsto x$ (a stack of coupling layers), and trains it
  by **maximum likelihood** using the exact change-of-variables. It does **not** learn a velocity field
  — there is no time variable inside $f_\theta$, no "wind." Flow matching instead learns a
  **time-dependent velocity field** $v_\theta(x,t)$ and trains it by plain **MSE arrow-regression**.
  Side by side, the two objectives are:
  $$
  \textbf{NF (L15):}\quad \max_\theta\ \mathbb{E}_{x\sim\text{data}}
      \Big[\log p_Z\!\big(f_\theta^{-1}(x)\big) + \log\big|\det J_{f_\theta^{-1}}(x)\big|\Big],
  $$
  $$
  \textbf{FM (this lecture):}\quad \min_\theta\ \mathbb{E}_{t,\,x_0,\,x_1}
      \big\lVert v_\theta(x_t,t) - (x_1 - x_0)\big\rVert^2 .
  $$
  The NF objective forces an invertible architecture and a Jacobian determinant; the FM objective has
  **no architectural constraint** and **no Jacobian** — that is the whole point. (After training, FM can
  still recover a likelihood via A4, so the L15 evaluation machinery still applies.)
- **Diffusion models (L17).** Diffusion can be seen as flow matching with a particular **Gaussian,
  curved** probability path (a VP/VE noise schedule) instead of our straight interpolant — a *different
  member of the same family*. Its probability-flow ODE is exactly a CNF with a velocity field, but
  diffusion parameterizes the **score** $\nabla\log p_t$ (equivalently, predicts the noise $\varepsilon$).
  The curved path is longer, so diffusion needs many more integration steps; in exchange the score gives
  an exact stochastic reverse process. Velocity, score, and denoiser are interconvertible for Gaussian
  paths, so an L17 $\varepsilon$-network and this lecture's $v_\theta$ encode the same information.

### References
1. Lipman, Chen, Ben-Hamu, Nickel, Le, *Flow Matching for Generative Modeling*, ICLR 2023. arXiv:2210.02747.
2. Liu, Gong, Liu, *Flow Straight and Fast (Rectified Flow)*, ICLR 2023. arXiv:2209.03003.
3. Albergo, Vanden-Eijnden, *Building Normalizing Flows with Stochastic Interpolants*, ICLR 2023. arXiv:2209.15571.
4. Tong et al., *Improving and Generalizing Flow-Based Generative Models with Minibatch Optimal Transport* (CFM / OT-CFM), TMLR 2024. arXiv:2302.00482.
5. Lipman et al., *Flow Matching Guide and Code*, 2024. arXiv:2412.06264.
6. Noé, Olsson, Köhler, Wu, *Boltzmann generators — sampling equilibrium states of many-body systems*, Science 365, eaaw1147 (2019).
7. Midgley et al., *Flow Annealed Importance Sampling Bootstrap (FAB)* (alanine dipeptide Boltzmann generator), ICLR 2023. arXiv:2208.01893.
8. Klein, Krämer, Noé, *Equivariant Flow Matching* (Boltzmann generators), NeurIPS 2023. arXiv:2306.15030.

---

*End of Lecture 16.*
